# Minimize a nonlinear function of two variables with an inequality constraint

Do imports.

In [ ]:
import numpy as np
from qpsolvers import solve_problem, Problem
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import sympy as sym

%matplotlib widget

# Suppress the display of very small numbers
np.set_printoptions(suppress=True)

## Stay inside the circle

Suppose we want to minimize the same function

$$ f(x) = (a - x_1)^2 + b (x_2 - x_1^2)^2 $$

as before, but now subject to the constraint that

$$ h(x) = \begin{bmatrix} (x_1 - c_1)^2 + (x_2 - c_2)^2 - r^2 \end{bmatrix} \leq 0. $$

Let's define the cost $f(x)$, the constraint $h(x)$, the gradient $\nabla f(x)$ of the cost, and the Jacobian $\mathbf{J}h(x)$ of the constraint.

In [ ]:
# Parameters
params = {
    'a': 1,
    'b': 1,
    'c_x1': -1,
    'c_x2': 0,
    'c_r': 1,
}

# Variables
x1, x2 = sym.symbols('x1, x2')

# Cost
f_sym = (params['a'] - x1)**2 + params['b'] * (x2 - x1**2)**2
f_batch = sym.lambdify([x1, x2], f_sym)
f = lambda x: f_batch(*x)

# Constraint (inequality)
h_sym = sym.Matrix([(x1 - params['c_x1'])**2 + (x2 - params['c_x2'])**2 - params['c_r']**2])
h_batch = sym.lambdify([x1, x2], h_sym)
h = lambda x: h_batch(*x).flatten()

# Gradient of cost
f_x_sym = sym.simplify(sym.Matrix([f_sym]).jacobian([x1, x2]).T)
f_x_batch = sym.lambdify([x1, x2], f_x_sym)
f_x = lambda x: f_x_batch(*x).flatten()

# Jacobian of constraint (row i of this matrix is the transpose
# of the gradient of h_i(x) with respect to x)
h_x_sym = sym.simplify(h_sym.jacobian([x1, x2]))
h_x_batch = sym.lambdify([x1, x2], h_x_sym)
h_x = lambda x: h_x_batch(*x)

Define the **Lagrangian function**

$$ L(x, \lambda) = f(x) + \lambda^\top h(x) $$

and compute both its gradient $\nabla_x L(x, \lambda)$ and its hessian $\mathbf{H}_x L(x, \lambda)$ with respect to $x$.

In [ ]:
# Lagrangian
l1 = sym.symbols('lambda_1')
L_sym = f_sym + sym.Matrix([l1]).dot(h_sym)
L = lambda x, l: sym.lambdify([x1, x2, l1], L_sym)(*x, *l)

# Gradient of Lagrangian with respect to x
L_x_sym = sym.Matrix([L_sym]).jacobian([x1, x2]).T
L_x_batch = sym.lambdify([x1, x2, l1], L_x_sym)
L_x = lambda x, l: L_x_batch(*x, *l)

# Hessian of Lagrangian with respect to x
L_xx_sym = L_x_sym.jacobian([x1, x2])
L_xx_batch = sym.lambdify([x1, x2, l1], L_xx_sym)
L_xx = lambda x, l: L_xx_batch(*x, *l)

# Display
display(Markdown(r'$L(x, \lambda)' + f' = {sym.latex(L_sym)}$'))
display(Markdown(r'$\nabla_x L(x, \lambda)' + f' = {sym.latex(L_x_sym)}$'))
display(Markdown(r'$\mathbf{H}_{x} L(x, \lambda)' + f' = {sym.latex(L_xx_sym)}$'))

Define a function that checks if a candidate pair $(\widehat{a}, \widehat{b})$ is *dominated* by every pair in
$$
\{ (a_1, b_1), \dotsc, (a_n, b_n) \}.
$$
What it means for $(\widehat{a}, \widehat{b})$ to be dominated by $(a_i, b_i)$ is that $a_i \leq \widehat{a}$ and $b_i \leq \widehat{b}$. See Definition 15.2 of Nocedal and Wright (Numerical Optimization).

In [ ]:
# Define function
def is_dominated(candidate_pair, pairs, verbose=False):
    """
    Returns True if candidate_pair is dominated by every pair in
    pairs, and False otherwise.
    """
    assert(len(candidate_pair) == 2)
    a_hat, b_hat = candidate_pair
    for pair in pairs:
        a, b = pair
        if (a <= a_hat) and (b <= b_hat):
            if verbose:
                print(f'    ({a:6.3e}, {b:6.3e}) dominates ({a_hat:6.3e}, {b_hat:6.3e})')
            return True
    return False

# Test function
assert(is_dominated([10., 0.], [[5., 0.], [15., 0.]], verbose=True))
assert(not is_dominated([1., 0.], [[5., 0.], [15., 0.]], verbose=True))
assert(not is_dominated([1., 1.], [[3., 2.], [2., 3.]], verbose=True))
assert(not is_dominated([1., 4.], [[3., 2.], [2., 3.]], verbose=True))
assert(is_dominated([4., 4.], [[3., 2.], [2., 3.]], verbose=True))

Solve full problem.

In [ ]:
# Choose initial guess and compute cost and residual
x = np.array([-1., 4.])
l = np.array([0.])
res = [
    np.linalg.norm(L_x(x, l), np.inf),
    np.linalg.norm(np.maximum(0, h(x)), np.inf),
]
cost = f(x)

# Create log to keep track of convergence
log = {
    'x': [x],
    'res': [res],
    'cost': [cost],
}

# Create list of pairs for filter method
pairs = [[f(x), np.linalg.norm(np.maximum(0, h(x)), np.inf)]]

# Choose parameters
max_iters = 500
max_inner_iters = 50
tol = 1e-8
rho = 0.5
delta = 1e-6
omega = 10.

# Iterate
alpha = None
H = None
mu = None
success = False
for i in range(max_iters):
    # Show progress
    mu_str = f' : mu = {mu:.2e}' if mu is not None else ''
    alpha_str = f' : alpha = {alpha:.2e}' if alpha is not None else ''
    print(f'{i:3d} : l_1 = {l[0]:7.4e} : |L_x| = {res[0]:11.8f} : |h| = {res[1]:11.8f} : f = {cost:7.4f}' + mu_str + alpha_str)

    # Check stopping condition (residuals)
    if (res[0] < tol) and (res[1] < tol):
        success = True
        if i == 0:
            print(f'success (initial guess satisfies necessary conditions for optimality)')
        else:
            print(f'success (converged at iteration {i})')
        break
    
    # Choose descent direction (Newton step with regularization)
    L_xx_val = L_xx(x, l)
    f_x_val = f_x(x)
    h_x_val = h_x(x)
    h_val = h(x)
    H = L_xx_val.copy()
    mu_iters = 0
    while True:
        try:
            # Add regularization to the Hessian
            if mu_iters == 0:
                mu = 0.
            else:
                mu = delta * (omega**mu_iters)
            H = L_xx_val + (mu * np.eye(len(x)))
            # Attempt a Cholesky factorization - if it fails, then H is not
            # positive definite and we need to add more regularization
            L_chol = np.linalg.cholesky(H)
            # Attempt to solve for the descent direction - if it fails, then
            # H is ill-conditioned and we need to add more regularization
            problem = Problem(P=H, q=f_x_val, G=h_x_val, h=-h_val)
            solution = solve_problem(problem, solver='proxqp', eps_abs=1e-8, eps_rel=1e-8)
            p_x = solution.x
            p_l = solution.z - l
            #
            # ^^^ Be careful to distinguish between solution.y (dual variable
            #     associated with equality constraints) and solution.z (dual
            #     variable associated with inequality constraints). Get this
            #     wrong and you will have trouble finding your mistake!
            #
            break
        except np.linalg.LinAlgError:
            mu_iters += 1
    
    # Apply backtracking line search (filter method)
    alpha = 1.
    no_progress = True
    for i_inner in range(max_inner_iters):
        if is_dominated([f(x + alpha * p_x), np.linalg.norm(np.maximum(0, h(x + alpha * p_x)), np.inf)], pairs, verbose=True):
            alpha *= rho
        else:
            no_progress = False
            break
    
    # Check stopping condition (no progress)
    if no_progress:
        print(f'failure (no progress at iteration {i})')
        break

    # Update guess
    x = x + alpha * p_x
    l = l + alpha * p_l
    res = [
        np.linalg.norm(L_x(x, l), np.inf),
        np.linalg.norm(np.maximum(0, h(x)), np.inf),
    ]
    cost = f(x)
    pairs.append([f(x), np.linalg.norm(np.maximum(0, h(x)), np.inf)])

    # Update log
    log['x'].append(x)
    log['res'].append(res)
    log['cost'].append(cost)

# Check if max iters was exceeded
if (not success) and (i == max_iters):
    print(f'failure (exceeded maximum number {max_iters} of iterations)')

# Clean log
for k in log.keys():
    log[k] = np.array(log[k])

Visualize solution.

In [ ]:
# Close any prior figures
plt.close('Solution path for inequality-constrained minimization (inside circle)')

# Extract solution path from log
x_iters = log['x']
res_iters = log['res']

# Figure
fig, (ax2d, ax_res_hradL, ax_res_h) = plt.subplots(
    1, 3, figsize=(10, 4), tight_layout=True,
    num='Solution path for inequality-constrained minimization (inside circle)',
)

# Cost
x1_bnds = [-3, 3]
x2_bnds = [-2, 6]
X1, X2 = np.meshgrid(
    np.linspace(*x1_bnds, 500),
    np.linspace(*x2_bnds, 500),
)
F = f_batch(X1, X2)
CS = ax2d.contour(X1, X2, F, levels=[0.1, 1., 5., 10., 20., 30.], norm='log', cmap='Blues_r')
ax2d.clabel(CS, fontsize=10)

# Constraint circle
circ = plt.Circle((params['c_x1'], params['c_x2']), params['c_r'], color='C1', fill=True, linewidth=0, zorder=2, alpha=0.5)
ax2d.add_patch(circ)

# Residuals
ax_res_hradL.semilogy(res_iters[:, 0], color='black', label=r'$\|\nabla_x L(x_k, \lambda_k)\|$', marker='.', markersize=6)
ax_res_h.plot(res_iters[:, 1], color='black', label=r'$\|\text{max}(0, h(x_k))\|$', marker='.', markersize=6)
if np.any(res_iters[:, 1] > 0):
    ax_res_h.set_yscale('log')

# Appearance
ax2d.grid()
ax2d.plot(x_iters[:, 0], x_iters[:, 1], 'r.-', markersize=6)
ax2d.plot(x_iters[:, 0], x_iters[:, 1], '.-', color='black', markersize=6)
ax2d.grid()
ax2d.set_xlabel(r'$x_1$')
ax2d.set_ylabel(r'$x_2$')
ax2d.set_xlim(*x1_bnds)
ax2d.set_ylim(*x2_bnds)
ax2d.set_aspect('equal')
ax_res_hradL.grid()
ax_res_hradL.legend()
ax_res_hradL.set_xlabel('number of iterations')
ax_res_h.grid()
ax_res_h.legend()
ax_res_h.set_xlabel('number of iterations')
plt.show()

## Stay outside the circle

Suppose we want to minimize the same function

$$ f(x) = (a - x_1)^2 + b (x_2 - x_1^2)^2 $$

as before, but now subject to the constraint that

$$ \begin{bmatrix} (x_1 - c_1)^2 + (x_2 - c_2)^2 - r^2 \end{bmatrix} \geq 0 $$

or equivalently that

$$ h(x) = - \begin{bmatrix} (x_1 - c_1)^2 + (x_2 - c_2)^2 - r^2 \end{bmatrix} \leq 0. $$

Let's define the cost $f(x)$, the constraint $h(x)$, the gradient $\nabla f(x)$ of the cost, and the Jacobian $\mathbf{J}h(x)$ of the constraint.

In [ ]:
# Parameters
params = {
    'a': 1,
    'b': 1,
    'c_x1': 0,
    'c_x2': 0,
    'c_r': 1,
}

# Variables
x1, x2 = sym.symbols('x1, x2')

# Cost
f_sym = (params['a'] - x1)**2 + params['b'] * (x2 - x1**2)**2
f_batch = sym.lambdify([x1, x2], f_sym)
f = lambda x: f_batch(*x)

# Constraint (inequality)
h_sym = - sym.Matrix([(x1 - params['c_x1'])**2 + (x2 - params['c_x2'])**2 - params['c_r']**2])
h_batch = sym.lambdify([x1, x2], h_sym)
h = lambda x: h_batch(*x).flatten()

# Gradient of cost
f_x_sym = sym.simplify(sym.Matrix([f_sym]).jacobian([x1, x2]).T)
f_x_batch = sym.lambdify([x1, x2], f_x_sym)
f_x = lambda x: f_x_batch(*x).flatten()

# Jacobian of constraint (row i of this matrix is the transpose
# of the gradient of h_i(x) with respect to x)
h_x_sym = sym.simplify(h_sym.jacobian([x1, x2]))
h_x_batch = sym.lambdify([x1, x2], h_x_sym)
h_x = lambda x: h_x_batch(*x)

Define the **Lagrangian function**

$$ L(x, \lambda) = f(x) + \lambda^\top h(x) $$

and compute both its gradient $\nabla_x L(x, \lambda)$ and its hessian $\mathbf{H}_x L(x, \lambda)$ with respect to $x$.

In [ ]:
# Lagrangian
l1 = sym.symbols('lambda_1')
L_sym = f_sym + sym.Matrix([l1]).dot(h_sym)
L = lambda x, l: sym.lambdify([x1, x2, l1], L_sym)(*x, *l)

# Gradient of Lagrangian with respect to x
L_x_sym = sym.Matrix([L_sym]).jacobian([x1, x2]).T
L_x_batch = sym.lambdify([x1, x2, l1], L_x_sym)
L_x = lambda x, l: L_x_batch(*x, *l)

# Hessian of Lagrangian with respect to x
L_xx_sym = L_x_sym.jacobian([x1, x2])
L_xx_batch = sym.lambdify([x1, x2, l1], L_xx_sym)
L_xx = lambda x, l: L_xx_batch(*x, *l)

# Display
display(Markdown(r'$L(x, \lambda)' + f' = {sym.latex(L_sym)}$'))
display(Markdown(r'$\nabla_x L(x, \lambda)' + f' = {sym.latex(L_x_sym)}$'))
display(Markdown(r'$\mathbf{H}_{x} L(x, \lambda)' + f' = {sym.latex(L_xx_sym)}$'))

Solve full problem.

Be careful! Increase `c_r` and decrease `tol` and see what happens. Why? (Hint: think about regularization.)

Be careful! Change the initial guess to `x = np.array([0., 0.])` and see what happens. Why? (Hint: think about the filter method. This is why most people who use filter methods also use a "trust region" - see Fletcher, Leyffer, and Toint (2002). If you try to "fix" this problem in a clever way, does your approach still work for the other minimization problems we have considered?)

These are examples of things that other people have thought carefully about in the design and implementation of existing solvers.

In [ ]:
# Choose initial guess and compute cost and residual
x = np.array([-1., 4.])
l = np.array([0.])
res = [
    np.linalg.norm(L_x(x, l), np.inf),
    np.linalg.norm(np.maximum(0, h(x)), np.inf),
]
cost = f(x)

# Create log to keep track of convergence
log = {
    'x': [x],
    'res': [res],
    'cost': [cost],
}

# Create list of pairs for filter method
pairs = [[f(x), np.linalg.norm(np.maximum(0, h(x)), np.inf)]]

# Choose parameters
max_iters = 500
max_inner_iters = 50
tol = 1e-8
rho = 0.5
delta = 1e-6
omega = 10.

# Iterate
alpha = None
H = None
mu = None
success = False
for i in range(max_iters):
    # Show progress
    mu_str = f' : mu = {mu:.2e}' if mu is not None else ''
    alpha_str = f' : alpha = {alpha:.2e}' if alpha is not None else ''
    print(f'{i:3d} : l_1 = {l[0]:7.4e} : |L_x| = {res[0]:11.8f} : |h| = {res[1]:11.8f} : f = {cost:7.4f}' + mu_str + alpha_str)

    # Check stopping condition (residuals)
    if (res[0] < tol) and (res[1] < tol):
        success = True
        if i == 0:
            print(f'success (initial guess satisfies necessary conditions for optimality)')
        else:
            print(f'success (converged at iteration {i})')
        break
    
    # Choose descent direction (Newton step with regularization)
    L_xx_val = L_xx(x, l)
    f_x_val = f_x(x)
    h_x_val = h_x(x)
    h_val = h(x)
    mu_iters = 0
    while True:
        try:
            # Add regularization to the Hessian
            if mu_iters == 0:
                mu = 0.
            else:
                mu = delta * (omega**mu_iters)
            H = L_xx_val + (mu * np.eye(len(x)))
            # Attempt a Cholesky factorization - if it fails, then H is not
            # positive definite and we need to add more regularization
            L_chol = np.linalg.cholesky(H)
            # Attempt to solve for the descent direction - if it fails, then
            # H is ill-conditioned and we need to add more regularization
            problem = Problem(P=H, q=f_x_val, G=h_x_val, h=-h_val)
            solution = solve_problem(problem, solver='proxqp', eps_abs=1e-8, eps_rel=1e-8)
            p_x = solution.x
            p_l = solution.z - l
            #
            # ^^^ Be careful to distinguish between solution.y (dual variable
            #     associated with equality constraints) and solution.z (dual
            #     variable associated with inequality constraints). Get this
            #     wrong and you will have trouble finding your mistake!
            #
            break
        except np.linalg.LinAlgError:
            mu_iters += 1
    
    # Apply backtracking line search (filter method)
    alpha = 1.
    no_progress = True
    for i_inner in range(max_inner_iters):
        if is_dominated([f(x + alpha * p_x), np.linalg.norm(np.maximum(0, h(x + alpha * p_x)), np.inf)], pairs, verbose=True):
            alpha *= rho
        else:
            no_progress = False
            break
    
    # Check stopping condition (no progress)
    if no_progress:
        print(f'failure (no progress at iteration {i})')
        break

    # Update guess
    x = x + alpha * p_x
    l = l + alpha * p_l
    res = [
        np.linalg.norm(L_x(x, l), np.inf),
        np.linalg.norm(np.maximum(0, h(x)), np.inf),
    ]
    cost = f(x)
    pairs.append([f(x), np.linalg.norm(np.maximum(0, h(x)), np.inf)])

    # Update log
    log['x'].append(x)
    log['res'].append(res)
    log['cost'].append(cost)

# Check if max iters was exceeded
if (not success) and (i == max_iters):
    print(f'failure (exceeded maximum number {max_iters} of iterations)')

# Clean log
for k in log.keys():
    log[k] = np.array(log[k])

Visualize solution.

In [ ]:
# Close any prior figures
plt.close('Solution path for inequality-constrained minimization (outside circle)')

# Extract solution path from log
x_iters = log['x']
res_iters = log['res']

# Figure
fig, (ax2d, ax_res_hradL, ax_res_h) = plt.subplots(
    1, 3, figsize=(10, 4), tight_layout=True,
    num='Solution path for inequality-constrained minimization (outside circle)',
)

# Cost
x1_bnds = [-3, 3]
x2_bnds = [-2, 6]
X1, X2 = np.meshgrid(
    np.linspace(*x1_bnds, 500),
    np.linspace(*x2_bnds, 500),
)
F = f_batch(X1, X2)
CS = ax2d.contour(X1, X2, F, levels=[0.1, 1., 5., 10., 20., 30.], norm='log', cmap='Blues_r')
ax2d.clabel(CS, fontsize=10)

# Constraint circle
circ = plt.Circle((params['c_x1'], params['c_x2']), params['c_r'], color='C1', fill=True, linewidth=0, zorder=2, alpha=0.5)
ax2d.add_patch(circ)

# Residuals
ax_res_hradL.semilogy(res_iters[:, 0], color='black', label=r'$\|\nabla_x L(x_k, \lambda_k)\|$', marker='.', markersize=6)
ax_res_h.plot(res_iters[:, 1], color='black', label=r'$\|\text{max}(0, h(x_k))\|$', marker='.', markersize=6)
if np.any(res_iters[:, 1] > 0):
    ax_res_h.set_yscale('log')

# Appearance
ax2d.grid()
ax2d.plot(x_iters[:, 0], x_iters[:, 1], 'r.-', markersize=6)
ax2d.plot(x_iters[:, 0], x_iters[:, 1], '.-', color='black', markersize=6)
ax2d.grid()
ax2d.set_xlabel(r'$x_1$')
ax2d.set_ylabel(r'$x_2$')
ax2d.set_xlim(*x1_bnds)
ax2d.set_ylim(*x2_bnds)
ax2d.set_aspect('equal')
ax_res_hradL.grid()
ax_res_hradL.legend()
ax_res_hradL.set_xlabel('number of iterations')
ax_res_h.grid()
ax_res_h.legend()
ax_res_h.set_xlabel('number of iterations')
plt.show()